Xây dựng model 


In [1]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error
import time
#Đọc file dữ liệu final_data.csv đã được làm sạch
df = pd.read_csv('final_data.csv')


In [2]:

print("--- KHỞI CHẠY CHIẾN DỊCH TỐI ƯU TOÀN DIỆN: ÉP RMSE < 0.5 ---")
start_time = time.time()
df['Date'] = pd.to_datetime(df['Date'])
split_date = df['Date'].max() - pd.Timedelta(days=56)
train_df = df[df['Date'] < split_date].reset_index(drop=True)
val_df = df[df['Date'] >= split_date].reset_index(drop=True)
drop_cols = ['Date', 'ItemCode', 'Quantity']
features = [col for col in df.columns if col not in drop_cols]
target = 'Quantity'
X_train, y_train = train_df[features], train_df[target]
X_val, y_val = val_df[features], val_df[target]
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val) 
cat_features = ['ABC_Class', 'dayofweek', 'month', 'is_weekend']
train_data = lgb.Dataset(X_train, label=y_train_log, categorical_feature=cat_features)
val_data = lgb.Dataset(X_val, label=y_val_log, reference=train_data, categorical_feature=cat_features)

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.015,         
    'num_leaves': 15,              
    'max_depth': 6,                 
    'min_data_in_leaf': 150,        
    'feature_fraction': 0.7,       
    'bagging_fraction': 0.7,
    'bagging_freq': 1,            
    'verbose': -1,
    'seed': 42                     
}
# Huấn luyện global model
global_model = lgb.train(
    params,
    train_data,
    num_boost_round=1500,           
    valid_sets=[train_data, val_data],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=100)
    ]
)

end_time = time.time()
print(f"Thời gian huấn luyện Global Model: {end_time - start_time:.2f} giây")
val_pred_log = global_model.predict(X_val, num_iteration=global_model.best_iteration)
val_df['pred'] = np.expm1(val_pred_log)
val_df['pred'] = np.clip(val_df['pred'], a_min=0, a_max=None) 
df_val_preds = val_df[['ItemCode', 'Date', 'Quantity']].copy()
df_val_preds['Raw_Pred'] = val_df['pred']
df_val_preds['Final_Pred'] = val_df['pred'] 
if 'rolling_mean_56' in X_val.columns:
    long_tail_mask = (X_val['ABC_Class'] == 2) & (X_val['rolling_mean_56'] < 0.01)
    df_val_preds.loc[long_tail_mask, 'Final_Pred'] = 0
df_val_preds.loc[X_val['dayofweek'] == 6, 'Final_Pred'] = 0
final_rmse = np.sqrt(mean_squared_error(df_val_preds['Quantity'], df_val_preds['Final_Pred']))
print("\n" + "="*50)
print(f" ĐIỂM RMSE SAU KHI TỐI ƯU TOÀN DIỆN: {final_rmse:.4f}")
print("="*50)

--- KHỞI CHẠY CHIẾN DỊCH TỐI ƯU TOÀN DIỆN: ÉP RMSE < 0.5 ---
Training until validation scores don't improve for 50 rounds
[100]	training's rmse: 0.159154	valid_1's rmse: 0.154059
Early stopping, best iteration is:
[128]	training's rmse: 0.157368	valid_1's rmse: 0.153689
Thời gian huấn luyện Global Model: 4.64 giây

 ĐIỂM RMSE SAU KHI TỐI ƯU TOÀN DIỆN: 0.8526


In [3]:
#Lưu dữ liệu dự đoán vào file CSV để sử dụng cho bước tiếp theo (xử lý hậu kì)
df_val_preds.to_csv('val_predictions.csv', index=False)

In [4]:
import numpy as np
import pandas as pd
import os

print("="*50)
print("TIẾN TRÌNH CHUYỂN ĐỔI FORMAT SUBMISSION TỰ ĐỘNG KHỚP DỮ LIỆU")
print("="*50)

# 1. Đọc file mẫu của BTC làm bộ khung bắt buộc
sample_sub_path = 'sample_submission.csv'
if not os.path.exists(sample_sub_path):
    raise FileNotFoundError(f"Không tìm thấy file '{sample_sub_path}'! Hãy upload file mẫu vào cùng thư mục Colab/Working.")

sub_template = pd.read_csv(sample_sub_path)
submission_data = val_df[['ItemCode', 'Date']].copy()
submission_data['Prediction'] = val_df['pred']
unique_dates = sorted(submission_data['Date'].unique())
print(f"-> Tìm thấy tổng cộng {len(unique_dates)} ngày trong tập dữ liệu dự báo của bạn.")

if len(unique_dates) == 56:
    val_dates = unique_dates[:28]   
    eval_dates = unique_dates[28:]  
else:
    val_dates = unique_dates[:28]
    eval_dates = []
    print("Cảnh báo: Tập dữ liệu của bạn không đủ 56 ngày, các ô trống của Evaluation sẽ tự gán bằng 0.")
val_map = pd.DataFrame({'Date': val_dates, 'F_col': [f'F{i}' for i in range(1, 29)], 'suffix': 'validation'})
if len(unique_dates) >= 56:
    val_dates = unique_dates[:28]
    eval_dates = unique_dates[28:56]
    if len(unique_dates) > 56:
        print(f"Dữ liệu có {len(unique_dates)} ngày, chỉ dùng 56 ngày chuẩn (bỏ {len(unique_dates)-56} ngày thừa).")
else:
    val_dates = unique_dates[:28]
    eval_dates = []
    print("Tập dữ liệu của bạn không đủ 56 ngày, các ô trống của Evaluation sẽ tự gán bằng 0.")

eval_map = pd.DataFrame({
    'Date': eval_dates,
    'F_col': [f'F{i}' for i in range(1, len(eval_dates) + 1)],
    'suffix': 'evaluation'
})
date_mapping = pd.concat([val_map, eval_map], ignore_index=True)
submission_mapped = submission_data.merge(date_mapping, on='Date', how='inner')
submission_mapped['item_clean'] = submission_mapped['ItemCode'].astype(str).str.replace('SKU-', '', case=False)
submission_mapped['id'] = 'SKU-' + submission_mapped['item_clean'] + '_' + submission_mapped['suffix']
submission_wide = submission_mapped.pivot_table(
    index='id', 
    columns='F_col', 
    values='Prediction', 
    aggfunc='first'
).reset_index()
submission_final = sub_template[['id']].merge(submission_wide, on='id', how='left')
f_columns = [f'F{i}' for i in range(1, 29)]
for col in f_columns:
    if col not in submission_final.columns:
        submission_final[col] = 0.0
submission_final[f_columns] = submission_final[f_columns].fillna(0)
submission_final[f_columns] = submission_final[f_columns].round(2)
submission_final[f_columns] = np.clip(submission_final[f_columns], a_min=0, a_max=None)
submission_final = submission_final[['id'] + f_columns]
print("\n" + "="*50)
print("KẾT QUẢ KIỂM TRA CUỐI CÙNG:")
print("="*50)
print(f" Kích thước file nộp bài: {submission_final.shape} (Chuẩn yêu cầu: (31944, 29))")
non_zero_rows = (submission_final[f_columns].sum(axis=1) > 0).sum()
print(f" Số lượng dòng chứa kết quả dự báo thực tế (> 0): {non_zero_rows} / {len(submission_final)}")
submission_final.to_csv('submission_format.csv', index=False)
print(f"\n✓ Đã lưu file hoàn chỉnh và sẵn sàng nộp bài: submission_format.csv")

TIẾN TRÌNH CHUYỂN ĐỔI FORMAT SUBMISSION TỰ ĐỘNG KHỚP DỮ LIỆU
-> Tìm thấy tổng cộng 57 ngày trong tập dữ liệu dự báo của bạn.
Cảnh báo: Tập dữ liệu của bạn không đủ 56 ngày, các ô trống của Evaluation sẽ tự gán bằng 0.
Dữ liệu có 57 ngày, chỉ dùng 56 ngày chuẩn (bỏ 1 ngày thừa).

KẾT QUẢ KIỂM TRA CUỐI CÙNG:
 Kích thước file nộp bài: (31944, 29) (Chuẩn yêu cầu: (31944, 29))
 Số lượng dòng chứa kết quả dự báo thực tế (> 0): 1560 / 31944

✓ Đã lưu file hoàn chỉnh và sẵn sàng nộp bài: submission_format.csv
